In [ ]:
# =========================
# Cell 1: Imports & Helpers
# =========================

# Core libraries
import numpy as np
import pandas as pd
import warnings
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# Configuration
DATA_PATHS = {
    'train': "/kaggle/input/drw-crypto-market-prediction/train.parquet",
    'test': "/kaggle/input/drw-crypto-market-prediction/test.parquet",
    'submission': "/kaggle/input/drw-crypto-market-prediction/sample_submission.csv",
    'best_time_training': "/kaggle/input/best-time-training-unique-model-01/datetime_index_Q_model_uniquemodel.csv",
}

# Memory optimization
def reduce_memory_usage(df):
    start_mem = df.memory_usage().sum() / 1024**3
    print(f"Starting memory: {start_mem:.2f} GB")
    
    for col in df.select_dtypes(include=['float']).columns:
        col_min = df[col].min()
        col_max = df[col].max()
        
        if col_min > np.finfo(np.float16).min and col_max < np.finfo(np.float16).max:
            df[col] = df[col].astype(np.float16)
        elif col_min > np.finfo(np.float32).min and col_max < np.finfo(np.float32).max:
            df[col] = df[col].astype(np.float32)
    
    end_mem = df.memory_usage().sum() / 1024**3
    reduction = 100 * (start_mem - end_mem) / start_mem
    print(f"Memory reduced to: {end_mem:.2f} GB ({reduction:.1f}% reduction)")
    return df

# Feature engineering
def add_features(df):
    # Original interactions
    df['bid_ask_interaction'] = df['bid_qty'] * df['ask_qty']
    df['bid_buy_interaction'] = df['bid_qty'] * df['buy_qty']
    df['bid_sell_interaction'] = df['bid_qty'] * df['sell_qty']
    df['ask_buy_interaction'] = df['ask_qty'] * df['buy_qty']
    df['ask_sell_interaction'] = df['ask_qty'] * df['sell_qty']

    df['volume_weighted_sell'] = df['sell_qty'] * df['volume']
    df['buy_sell_ratio'] = df['buy_qty'] / (df['sell_qty'] + 1e-10)
    df['selling_pressure'] = df['sell_qty'] / (df['volume'] + 1e-10)
    df['log_volume'] = np.log1p(df['volume'])

    df['effective_spread_proxy'] = np.abs(df['buy_qty'] - df['sell_qty']) / (df['volume'] + 1e-10)
    df['bid_ask_imbalance'] = (df['bid_qty'] - df['ask_qty']) / (df['bid_qty'] + df['ask_qty'] + 1e-10)
    df['order_flow_imbalance'] = (df['buy_qty'] - df['sell_qty']) / (df['buy_qty'] + df['sell_qty'] + 1e-10)
    df['liquidity_ratio'] = (df['bid_qty'] + df['ask_qty']) / (df['volume'] + 1e-10)
    
    # New microstructure features
    df['net_order_flow'] = df['buy_qty'] - df['sell_qty']
    df['normalized_net_flow'] = df['net_order_flow'] / (df['volume'] + 1e-10)
    df['buying_pressure'] = df['buy_qty'] / (df['volume'] + 1e-10)
    df['volume_weighted_buy'] = df['buy_qty'] * df['volume']
    
    df['total_depth'] = df['bid_qty'] + df['ask_qty']
    df['depth_imbalance'] = (df['bid_qty'] - df['ask_qty']) / (df['total_depth'] + 1e-10)
    df['relative_spread'] = np.abs(df['bid_qty'] - df['ask_qty']) / (df['total_depth'] + 1e-10)
    df['log_depth'] = np.log1p(df['total_depth'])
    
    df['kyle_lambda'] = np.abs(df['net_order_flow']) / (df['volume'] + 1e-10)
    df['flow_toxicity'] = np.abs(df['order_flow_imbalance']) * df['volume']
    df['aggressive_flow_ratio'] = (df['buy_qty'] + df['sell_qty']) / (df['total_depth'] + 1e-10)
    
    df['volume_depth_ratio'] = df['volume'] / (df['total_depth'] + 1e-10)
    df['activity_intensity'] = (df['buy_qty'] + df['sell_qty']) / (df['volume'] + 1e-10)
    df['log_buy_qty'] = np.log1p(df['buy_qty'])
    df['log_sell_qty'] = np.log1p(df['sell_qty'])
    df['log_bid_qty'] = np.log1p(df['bid_qty'])
    df['log_ask_qty'] = np.log1p(df['ask_qty'])
    
    df['realized_spread_proxy'] = 2 * np.abs(df['net_order_flow']) / (df['volume'] + 1e-10)
    df['price_impact_proxy'] = df['net_order_flow'] / (df['total_depth'] + 1e-10)
    df['quote_volatility_proxy'] = np.abs(df['depth_imbalance'])
    
    df['flow_depth_interaction'] = df['net_order_flow'] * df['total_depth']
    df['imbalance_volume_interaction'] = df['order_flow_imbalance'] * df['volume']
    df['depth_volume_interaction'] = df['total_depth'] * df['volume']
    df['buy_sell_spread'] = np.abs(df['buy_qty'] - df['sell_qty'])
    df['bid_ask_spread'] = np.abs(df['bid_qty'] - df['ask_qty'])
    
    df['trade_informativeness'] = df['net_order_flow'] / (df['bid_qty'] + df['ask_qty'] + 1e-10)
    df['execution_shortfall_proxy'] = df['buy_sell_spread'] / (df['volume'] + 1e-10)
    df['adverse_selection_proxy'] = df['net_order_flow'] / (df['total_depth'] + 1e-10) * df['volume']
    
    df['fill_probability'] = df['volume'] / (df['buy_qty'] + df['sell_qty'] + 1e-10)
    df['execution_rate'] = (df['buy_qty'] + df['sell_qty']) / (df['total_depth'] + 1e-10)
    df['market_efficiency'] = df['volume'] / (df['bid_ask_spread'] + 1e-10)
    
    df['sqrt_volume'] = np.sqrt(df['volume'])
    df['sqrt_depth'] = np.sqrt(df['total_depth'])
    df['volume_squared'] = df['volume'] ** 2
    df['imbalance_squared'] = df['order_flow_imbalance'] ** 2
    
    df['bid_ratio'] = df['bid_qty'] / (df['total_depth'] + 1e-10)
    df['ask_ratio'] = df['ask_qty'] / (df['total_depth'] + 1e-10)
    df['buy_ratio'] = df['buy_qty'] / (df['buy_qty'] + df['sell_qty'] + 1e-10)
    df['sell_ratio'] = df['sell_qty'] / (df['buy_qty'] + df['sell_qty'] + 1e-10)
    
    df['liquidity_consumption'] = (df['buy_qty'] + df['sell_qty']) / (df['total_depth'] + 1e-10)
    df['market_stress'] = df['volume'] / (df['total_depth'] + 1e-10) * np.abs(df['order_flow_imbalance'])
    df['depth_depletion'] = df['volume'] / (df['bid_qty'] + df['ask_qty'] + 1e-10)
    
    df['net_buying_ratio'] = df['net_order_flow'] / (df['volume'] + 1e-10)
    df['directional_volume'] = df['net_order_flow'] * np.log1p(df['volume'])
    df['signed_volume'] = np.sign(df['net_order_flow']) * df['volume']
    
    return df.replace([np.inf, -np.inf], 0).fillna(0)

def create_aggregated_features(df, feature_list, prefix):
    df[f'{prefix}_sum'] = df[feature_list].sum(axis=1)
    df[f'{prefix}_mean'] = df[feature_list].mean(axis=1)
    df[f'{prefix}_median'] = df[feature_list].median(axis=1)
    df[f'{prefix}_max'] = df[feature_list].max(axis=1)
    df[f'{prefix}_min'] = df[feature_list].min(axis=1)
    df[f'{prefix}_std'] = df[feature_list].std(axis=1)
    return df

def extract_time_features(df):
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute
    df['dayofweek'] = df['timestamp'].dt.dayofweek
    df['Y_M_D_H'] = df['timestamp'].dt.strftime('%Y-%m-%d-%H')
    df['Y_M_D_H_M'] = df['timestamp'].dt.strftime('%Y-%m-%d-%H-%M')
    return df

In [ ]:
# ==============
# CELL 2
# Feature lists + main()
# ==============

Negative_features_list = ['X563', 'X560', 'X486', 'X215', 'X7', 'X290', 'X581', 'X492', 'X569', 'X590', 'X201', 'X625', 'X247', 'X283', 'X261', 'X626', 'X193', 'X545', 'X195', 'X291', 'X703', 'X243', 'X297', 'X38', 'X490', 'X252', 'X42', 'X41', 'X254', 'X706', 'X237', 'X40', 'X16', 'X331', 'X47', 'X571', 'X302', 'X711', 'X280', 'activity_intensity', 'X412', 'X580', 'X54', 'X679', 'X245', 'X700', 'X622', 'X646', 'X223', 'X608', 'X695', 'X48', 'X319', 'X621', 'X125', 'X539', 'X259', 'X651', 'X648', 'X278', 'X682', 'X441', 'X5', 'X378', 'X565', 'X343', 'X654', 'X629', 'X14', 'X448', 'ask_qty', 'bid_qty', 'X95', 'X127', 'X463', 'X698', 'X482', 'X123', 'X46', 'X488', 'X572', 'X521', 'X30', 'X562', 'X421', 'X578', 'X380', 'X303', 'X171', 'X731', 'X452', 'X690', 'X221', 'X226', 'X456', 'X32', 'X643', 'X115', 'X676', 'X217', 'X602', 'X329', 'X765', 'X507', 'X722', 'X708', 'X179', 'X636', 'X210', 'X670', 'X767', 'X416', 'X218', 'X317', 'X204', 'X461', 'X250', 'X208', 'X159', 'X385', 'X198', 'X450', 'X28', 'X457', 'X400', 'X373', 'X272', 'X77', 'X154', 'X777', 'X256', 'X203', 'X20', 'X326', 'X407', 'X476', 'X274', 'X424', 'X33', 'X267', 'X443', 'X499', 'X113', 'X547', 'X382', 'X554', 'X583', 'X76', 'X639', 'X81']

Positive_features_list = ['X557', 'X566', 'X485', 'X194', 'X575', 'X584', 'X587', 'X493', 'X216', 'X44', 'X253', 'X627', 'X50', 'X707', 'X36', 'X6', 'X287', 'X262', 'X551', 'X244', 'X8', 'X286', 'X214', 'X236', 'X624', 'X623', 'X483', 'X337', 'X15', 'X240', 'X200', 'X289', 'X39', 'X699', 'X533', 'X222', 'X260', 'X325', 'X647', 'X299', 'X285', 'X650', 'X577', 'X284', 'X559', 'X43', 'X202', 'X702', 'X251', 'X704', 'X246', 'X418', 'X710', 'X683', 'X55', 'X620', 'X694', 'X298', 'X31', 'X292', 'X574', 'X239', 'X605', 'X288', 'X494', 'X121', 'X449', 'X462', 'X89', 'X696', 'X489', 'total_depth', 'X678', 'X165', 'X628', 'X406', 'X675', 'X568', 'X294', 'X384', 'X131', 'X481', 'X374', 'X119', 'X56', 'X133', 'X442', 'X277', 'X422', 'X655', 'X9', 'X451', 'X455', 'X209', 'X440', 'X652', 'X379', 'X323', 'X279', 'X611', 'X735', 'X766', 'X527', 'X205', 'X129', 'X644', 'X21', 'X224', 'X196', 'X642', 'X313', 'X293', 'X427', 'X258', 'X586', 'X225', 'X726', 'X691', 'X219', 'X335', 'X4', 'X635', 'X506', 'X415', 'X173', 'X458', 'X117', 'X372', 'X197', 'X27', 'X281', 'X672', 'X275', 'X500', 'X775', 'X469', 'X680', 'X599', 'X35', 'X29', 'X477', 'X401', 'X330', 'X160', 'X370', 'X266', 'X408', 'X640', 'X87', 'X70']

Features = ['X455', 'X557', 'X6', 'X259', 'X627', 'X450', 'X48', 'X44', 'X448', 'X291', 'X563', 'X218', 'X574', 'X695', 'X41', 'X418', 'X644', 'X621', 'X648', 'X451', 'X443', 'X643', 'Negative_features_10_sum', 'X722', 'X290', 'X200', 'Positive_features_10_sum', 'X458', 'X286', 'X571', 'X640', 'X559', 'X625', 'X222', 'X201', 'X214', 'X193', 'X639', 'X197', 'X31', 'X323', 'X456', 'X319', 'X278', 'X260', 'X251', 'X651', 'X5', 'X652', 'X285', 'X584', 'X317', 'X412', 'X647', 'Positive_features_100_std', 'X42', 'X440', 'X225', 'X575', 'X203', 'X683', 'X289', 'X204', 'X35', 'X562', 'X566', 'X605', 'X33', 'X293', 'Negative_features_5_sum', 'X219', 'X45', 'X587', 'X36', 'X16', 'X449', 'X569', 'X287', 'X246', 'X56', 'X726', 'X8', 'X379', 'X292', 'X679', 'X507', 'X406', 'Negative_features_150_std', 'X165', 'X329', 'X376', 'X15', 'X608', 'X626', 'X678', 'X628', 'X696', 'X457', 'X539', 'X469', 'X442', 'Positive_features_5_sum', 'Positive_features_60_std', 'Positive_features_90_std', 'X580', 'X694', 'X491', 'X258', 'X454', 'X46', 'X196', 'X123', 'X765', 'X370', 'X215', 'Negative_features_100_std', 'X331', 'X125', 'X119', 'X254', 'X279', 'X4', 'X298', 'X551', 'X768', 'X675', 'X226', 'X223', 'X43', 'X568', 'X444', 'X337', 'X320', 'Negative_features_60_std', 'X335', 'X581', 'X623', 'X294', 'X325', 'X572', 'X554', 'X416', 'X486', 'X602', 'X484', 'X577', 'X735', 'X288', 'X250', 'X691', 'X776', 'X766', 'X113', 'X611', 'X635', 'X666', 'X401', 'X489', 'X326', 'X699', 'X718', 'X636', 'X767', 'X476', 'Negative_features_70_std', 'X154', 'X221', 'X706', 'X506', 'X470', 'Positive_features_30_sum', 'X34', 'X21', 'X284', 'X38', 'X245', 'X700', 'X620', 'X415', 'X407', 'Positive_features_90_median', 'X32', 'X244', 'X670', 'X95', 'X343', 'X299', 'X129', 'X421', 'X482', 'X14', 'X668', 'X704', 'X9', 'X216', 'X77', 'X247', 'X155', 'Negative_features_90_std', 'Negative_features_50_sum', 'X477', 'X642', 'X465', 'X160', 'X373', 'X629', 'X777', 'X609', 'X724', 'X422', 'X672', 'X441', 'X302', 'X646', 'X527', 'X707', 'X71', 'X39', 'X194', 'X728', 'X410', 'X682', 'Positive_features_70_std', 'X49', 'X127', 'X198', 'X229', 'X579', 'X599', 'X622', 'X601', 'X461', 'fill_probability', 'X121', 'Positive_features_20_std', 'X676', 'Negative_features_30_sum', 'X573', 'X720', 'X50', 'X209', 'X690', 'X173', 'X89', 'X313', 'X439', 'X275', 'X171', 'X20', 'X30', 'X708', 'X237', 'X710', 'X217', 'X488', 'Negative_features_50_std', 'X133', 'X404', 'X281', 'X427', 'X28', 'X70', 'X650', 'X311', 'X731', 'X764', 'X385', 'X274', 'X51', 'Negative_features_150_median', 'X205', 'X283', 'X266', 'X408', 'X424', 'X610', 'X277', 'X261', 'X362', 'X230', 'X256', 'X280', 'X624', 'X447', 'X303', 'X175', 'X485', 'X462', 'X262', 'X115', 'Positive_features_50_sum', 'X117', 'Negative_features_30_std', 'X775', 'X565', 'X27', 'Negative_features_80_std', 'X377', 'X400', 'X272', 'X698', 'X578', 'Negative_features_60_sum', 'X159', 'X664', 'X269', 'X87', 'X680', 'X179', 'X508', 'X481', 'X499', 'X114', 'X545', 'X604', 'Positive_features_40_median', 'Positive_features_50_std', 'X267', 'X253', 'X459', 'X118', 'X547', 'X76', 'X464', 'Positive_features_150_std', 'X479', 'X295', 'X161', 'X192', 'X7', 'X316', 'Positive_features_90_max', 'X23', 'X548', 'X494', 'X500', 'X174', 'X521', 'X533', 'X655', 'X236', 'X24', 'ask_ratio', 'X382', 'X492', 'X380', 'X368', 'X29', 'X242', 'X148', 'X463', 'X238', 'X662', 'X598', 'X149', 'X124', 'X81', 'Negative_features_70_sum', 'X606', 'Negative_features_40_sum', 'X243', 'X723', 'X169', 'X314', 'X157', 'X84', 'X692', 'Negative_features_5_mean', 'X402', 'X328', 'X612', 'X613', 'X252', 'X769', 'Positive_features_5_mean', 'X210', 'X322', 'X235', 'X334', 'X271', 'X656', 'X409', 'Positive_features_5_std', 'X395', 'X255', 'X213', 'X107', 'X498', 'X446', 'X338', 'X530', 'X600', 'X112', 'X151', 'X583', 'X228', 'X78', 'X384', 'X372', 'X139', 'Negative_features_80_sum', 'X371', 'sell_ratio', 'X211', 'X120', 'X341', 'net_order_flow', 'X550', 'X90', 'Positive_features_40_std', 'X300', 'X667', 'X344', 'X40', 'X65', 'X497', 'X413', 'X490', 'X483', 'Negative_features_40_median', 'X212', 'X398', 'X73', 'X231', 'buy_qty', 'X734', 'Positive_features_30_std', 'Positive_features_80_max', 'X475', 'X163', 'X386', 'X542', 'X472', 'Negative_features_90_sum', 'X619', 'X282', 'activity_intensity', 'X576', 'X336', 'X241', 'X560', 'X375', 'X417', 'X333', 'X330', 'Positive_features_50_median', 'X590', 'Negative_features_10_mean', 'X240', 'Negative_features_40_std']

def main():
    train_df = pd.read_parquet(DATA_PATHS['train'])
    test_df = pd.read_parquet(DATA_PATHS['test'])
    sample_submission = pd.read_csv(DATA_PATHS['submission'])
    best_timeseries = pd.read_csv(DATA_PATHS['best_time_training'])

    train_df = reduce_memory_usage(train_df)
    test_df = reduce_memory_usage(test_df)

    train_df = add_features(train_df)
    test_df = add_features(test_df)

    for i in [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 150]:
        train_df = create_aggregated_features(train_df, Negative_features_list[:i], f"Negative_features_{i}")
        test_df = create_aggregated_features(test_df, Negative_features_list[:i], f"Negative_features_{i}")
        train_df = create_aggregated_features(train_df, Positive_features_list[:i], f"Positive_features_{i}")
        test_df = create_aggregated_features(test_df, Positive_features_list[:i], f"Positive_features_{i}")

    train_df = train_df.reset_index().rename(columns={'index': 'timestamp'})
    train_df = extract_time_features(train_df)

    best_time_training = pd.to_datetime(best_timeseries.values.ravel()).strftime('%Y-%m-%d-%H-%M')
    train_clean = train_df.loc[train_df["Y_M_D_H_M"].isin(best_time_training)]

    valid_features = [f for f in Features if f in train_clean.columns]
    print(f"Using {len(valid_features)} features out of {len(Features)}")

    X_train = train_clean[valid_features].values
    y_train = train_clean["label"]
    X_test = test_df[valid_features].values

    model = LinearRegression()
    model.fit(X_train, y_train)
    test_df["prediction"] = model.predict(X_test)

    sample_submission["prediction"] = test_df["prediction"].values
    sample_submission.to_csv('submission.csv', index=False)
    print("✅ Submission saved to submission.csv")

    return train_df, train_clean, test_df, model, X_train, y_train, X_test


if __name__ == "__main__":
    train_df, train_clean, test_df, model, X_train, y_train, X_test = main()

In [ ]:
# ==============
# CELL 3
# Quick EDA
# ==============

print(train_df.shape)
print(train_clean.shape)
display(train_df.head())
train_clean['label'].hist(bins=50)

## 📊 Label Distribution

The target variable (`label`) represents **future crypto price movement** - essentially what the model needs to predict.

**Key observations:**
- Distribution is tightly centered at **0**, meaning most time periods have near-zero price movement - the signal is very subtle.
- The slight **right skew** indicates more extreme positive movements than negative ones.
- The heavy peak around 0 makes this a **hard prediction problem** - most rows carry very little directional signal.
- Outliers beyond ±2 represent the high-value moments the model needs to capture most.

In [ ]:
### Cell 4

from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

y_pred = model.predict(X_train)
mse = mean_squared_error(y_train, y_pred)
print("Train MSE:", mse)

plt.figure(figsize=(6,6))
plt.scatter(y_train, y_pred, s=1, alpha=0.3)
plt.xlabel("True")
plt.ylabel("Predicted")
plt.title("Train fit")
plt.show()

## 📈 Label Distribution Deep Dive

A four-panel breakdown of the target variable using interactive Plotly charts.

**Histogram (top left):** Confirms the near-normal distribution with mean near zero. The dashed red line marks the mean; orange dotted lines mark ±1 standard deviation - roughly 68% of all labels fall within this band.

**Box Plot (top right):** The tight IQR (interquartile range) with many outlier dots above and below shows the distribution has **heavy tails** - extreme movements are real but rare.

**CDF (bottom left):** The S-curve shape shows ~50% of labels are below 0 (slight negative median). The steep rise near 0 confirms the concentration of near-zero values.

**Stats Table (bottom right):** The full numeric summary. Note the skewness and kurtosis values - kurtosis > 3 confirms **leptokurtic** behavior (fatter tails than a normal distribution), which is typical of financial return data.

In [ ]:
# ==============
# CELL 5
# GasMan: Top Feature Correlations with Label
# ==============

import plotly.express as px
from plotly.offline import iplot
import plotly.io as pio
pio.renderers.default = "iframe"

valid_features = [f for f in Features if f in train_clean.columns]

corr = train_clean[valid_features + ['label']].corr()['label'].drop('label')
corr_sorted = corr.abs().sort_values(ascending=False).head(30)
corr_vals = corr[corr_sorted.index]

fig = px.bar(
    x=corr_vals.values,
    y=corr_vals.index,
    orientation='h',
    color=corr_vals.values,
    color_continuous_scale='RdBu',
    title='Top 30 Features by Correlation with Label',
    labels={'x': 'Correlation', 'y': 'Feature'},
    template='plotly_dark',
    height=700
)
fig.update_layout(
    title_font=dict(size=18, color='#00d4ff'),
    yaxis=dict(autorange='reversed')
)
iplot(fig)

## 🔗 Feature Correlation with Label

This chart ranks the **top 30 features** by their absolute correlation with the target label.

**Key observations:**
- No single feature dominates with a very high correlation - the highest values are likely in the **0.05–0.20** range, which is normal for crypto microstructure data.
- Features in **blue** (positive correlation) move in the same direction as the label - rising order flow, buying pressure, etc.
- Features in **red** (negative correlation) move inversely - selling pressure, ask-side imbalance, etc.
- The mix of `X`-prefixed anonymous features and our **engineered microstructure features** (e.g. `order_flow_imbalance`, `net_order_flow`, `fill_probability`) tells us which signal types matter most.
- Features with near-zero correlation can potentially be dropped to reduce noise.

In [ ]:
# ==============
# CELL 6
# GasMan: Residual Analysis
# ==============

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

residuals = y_train.values - model.predict(X_train)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Residuals Distribution", "Residuals vs Predicted")
)

# Residual histogram
fig.add_trace(go.Histogram(
    x=residuals,
    nbinsx=100,
    marker_color='#00d4ff',
    name='Residuals',
    opacity=0.8
), row=1, col=1)

# Residuals vs predicted scatter
fig.add_trace(go.Scatter(
    x=model.predict(X_train),
    y=residuals,
    mode='markers',
    marker=dict(size=2, color='#ff6b6b', opacity=0.3),
    name='Residuals vs Pred'
), row=1, col=2)

fig.add_hline(y=0, line_dash='dash', line_color='white', row=1, col=2)

fig.update_layout(
    title='GasMan – Residual Analysis (Baseline LR)',
    template='plotly_dark',
    height=450,
    title_font=dict(size=18, color='#00d4ff'),
    showlegend=False
)
fig.show()

print(f"Mean residual:   {residuals.mean():.6f}")
print(f"Std residual:    {residuals.std():.6f}")
print(f"Max residual:    {residuals.max():.6f}")
print(f"Min residual:    {residuals.min():.6f}")

## 🔍 Residual Analysis – Baseline Linear Regression

Residuals are the difference between the **true label** and the **model's prediction**. A well-behaved model should have residuals that look like white noise centered at zero.

**Residual Histogram (left):** The near-normal shape centered at 0 is a good sign - no systematic bias in the baseline LR model.

**Residuals vs Predicted (right):** Ideally this should look like a flat random cloud around the zero line. Any **funnel shape** (heteroscedasticity) or **curve** would indicate the model is missing structure. Patterns here point to where a more powerful model (XGBoost, RF) could improve.

**What this means for R&D:** The residuals at the extremes (large positive/negative predictions) tend to have the largest errors - these are exactly the high-value trading moments. A tree-based model may capture these tails better than Linear Regression.

In [ ]:
# ==============
# CELL 7
# GasMan: Volume vs Label
# ==============

sample = train_clean.sample(5000, random_state=42)

fig = px.scatter(
    sample,
    x='log_volume',
    y='label',
    color='order_flow_imbalance',
    color_continuous_scale='RdBu',
    opacity=0.5,
    title='Log Volume vs Label (colored by Order Flow Imbalance)',
    labels={'log_volume': 'Log Volume', 'label': 'Label'},
    template='plotly_dark',
    height=500
)
fig.update_layout(title_font=dict(size=18, color='#00d4ff'))
fig.show()

## 📉 Log Volume vs Label (colored by Order Flow Imbalance)

This scatter explores the relationship between **trading volume** and **price movement direction**, colored by **order flow imbalance** (buy pressure minus sell pressure, normalized).

**Key observations:**
- Most activity is clustered in the **mid-volume range** (log volume 3–6), with labels tightly around 0.
- **Blue dots** (positive order flow imbalance = more buying than selling) tend to cluster slightly **above** the zero label line - confirming that buy-side pressure is a positive signal.
- **Red dots** (negative imbalance = sell pressure) tend to cluster **below** zero - confirming selling pressure predicts downward movement.
- **High volume outliers** (log volume > 7) show more extreme label values in both directions - large volume events are associated with larger price moves.
- This validates our microstructure feature engineering: `order_flow_imbalance` and `log_volume` carry real predictive signal.

# DRW Crypto Market Prediction - GasMan R&D Notebook

## Attribution & Credits

This notebook is a **research and development fork** built on top of the original 2nd place solution from the [DRW Crypto Market Prediction](https://www.kaggle.com/competitions/drw-crypto-market-prediction) competition hosted on Kaggle.

**Original Notebook:**
- Author: [Younes Eloi (ELOIARMYOUNES)](https://www.kaggle.com/eloiarmyounes)
- Notebook: [DRW: 2nd Place in the Private LB](https://www.kaggle.com/code/eloiarmyounes/drw-2nd-place-in-the-private-lb)
- Final standing: **2nd place on the private leaderboard**

**Competition:**
- Name: DRW Crypto Market Prediction
- Host: [DRW](https://drw.com) via Kaggle
- Objective: Develop a model capable of predicting crypto future price movements using anonymized microstructure features
- Link: [kaggle.com/competitions/drw-crypto-market-prediction](https://www.kaggle.com/competitions/drw-crypto-market-prediction)



## About This Fork (GasMan R&D)

**Author:** [gastondana](https://www.kaggle.com/gastondana)

This notebook extends the original solution with:
- Restructured pipeline split into clearly labeled cells
- Interactive Plotly visualizations (label distribution, feature correlations, residual analysis, volume vs label)
- Microstructure feature engineering additions (Kyle Lambda, flow toxicity, market stress, signed volume, etc.)
- R&D analysis layer with markdown documentation for each output
- Model comparison framework (LR baseline vs RF vs XGB)

> All original feature engineering, feature lists, time-filtering logic, and model pipeline are credited to the original author. Extensions and analysis are original contributions.
